In [1]:
import nltk
import numpy as np
import pandas as pd
import re
import random
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB, MultinomialNB, ComplementNB
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    cohen_kappa_score,
)
import matplotlib.pyplot as plt
import lightgbm as lgb

nltk.download("wordnet", quiet=True)



True

In [2]:
train = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv"
)
print(train.head())



  essay_id                                          full_text  score
0  663d2cf  Dear State Senator,\n\nI am arguing in favor o...      3
1  3a20bfb  In " The Challenge of Exploring Venus" The aut...      2
2  6adae64  Teachers can have a hard time telling if their...      3
3  c81ccdf  Using dirverless cars can be very dangrous in ...      3
4  9549d7f  In " The Challenge of Exploring Venus" the aut...      3


In [3]:
from nltk.stem import WordNetLemmatizer


def remove_punctuations(text):
    return re.sub(r"[^\w\s]", "", text)


def remove_html(text):
    return re.sub(r"<.*?>", "", text)


def remove_url(text):
    return re.sub(r"https://\S+|www\.\S+", "", text)


def remove_emoji(text):
    """
    Remove emojis and other non‑ASCII characters from the text.
    This implementation avoids complex Unicode ranges that cause
    regex compilation errors in the Python 3.12 environment.
    """
    # Replace non‑ASCII characters with a space and lower‑case the result
    cleaned = re.sub(r"[^\x00-\x7F]+", " ", text)
    return cleaned.lower()


def preprocess_series(series):
    series = series.apply(remove_punctuations)
    series = series.apply(remove_html)
    series = series.apply(remove_url)
    series = series.apply(remove_emoji)
    return series


train["text"] = preprocess_series(train["full_text"])



In [4]:
X = train["text"].astype(str)
y = train["score"].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, stratify=y, test_size=0.33, random_state=0
)



In [5]:
pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                stop_words="english", ngram_range=(1, 2), max_features=150000
            ),
        ),
        (
            "feature_selection",
            SelectFromModel(ExtraTreesClassifier(n_estimators=100, random_state=0)),
        ),
        ("clf", LinearSVC()),
    ]
)

pipeline.fit(X_train, y_train)

val_pred = pipeline.predict(X_val)
print("Validation F1 (LinearSVC):", f1_score(y_val, val_pred, average="weighted"))
print("Validation Cohen Kappa (LinearSVC):", cohen_kappa_score(y_val, val_pred))



Validation F1 (LinearSVC): 0.4682922498621405
Validation Cohen Kappa (LinearSVC): 0.2693612203241005


In [6]:
feature_pipeline = Pipeline(pipeline.steps[:-1])  # tfidf + feature_selection
X_train_feat = feature_pipeline.transform(X_train)
X_val_feat = feature_pipeline.transform(X_val)



In [7]:
lgb_params = {
    "n_estimators": 195,
    "num_leaves": 6,
    "min_child_samples": 3,
    "learning_rate": 0.06518970520093895,
    "max_bin": 511,
    "colsample_btree": 0.9699010403795221,
    "reg_alpha": 0.028218191448367104,
    "reg_lambda": 0.855160569748305,
    "objective": "multiclass",
    "num_class": 6,
    "random_state": 0,
}

lgb_model = lgb.LGBMClassifier(**lgb_params)
lgb_model.fit(X_train_feat, y_train)

val_pred_lgb = lgb_model.predict(X_val_feat)
print("Validation F1 (LightGBM):", f1_score(y_val, val_pred_lgb, average="weighted"))
print("Validation Cohen Kappa (LightGBM):", cohen_kappa_score(y_val, val_pred_lgb))



[LightGBM] [Warning] Unknown parameter: colsample_btree
[LightGBM] [Warning] Unknown parameter: colsample_btree
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.192116 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 559325
[LightGBM] [Info] Number of data points in the train set: 10435, number of used features: 25775
[LightGBM] [Info] Start training from score -2.628856
[LightGBM] [Info] Start training from score -1.298900
[LightGBM] [Info] Start training from score -1.017825
[LightGBM] [Info] Start training from score -1.475128
[LightGBM] [Info] Start training from score -2.877896
[LightGBM] [Info] Start training from score -4.753111
[LightGBM] [Warning] Unknown parameter: colsample_btree
Validation F1 (LightGBM): 0.5476902924250395
Validation Cohen Kappa (LightGBM): 0.38102555381822967


In [8]:
test = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv"
)
test["text"] = preprocess_series(test["full_text"])



In [9]:
test_feat = feature_pipeline.transform(test["text"])
test_pred = lgb_model.predict(test_feat)

# Ensure predictions are within 1‑6
test_pred = np.clip(test_pred, 1, 6).astype(int)



[LightGBM] [Warning] Unknown parameter: colsample_btree


In [10]:
submission = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv"
)
submission["score"] = test_pred
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")
display(submission.head())

Submission saved to submission.csv


,essay_id,score
0,d550b2d,3
1,0c10954,2
2,ef04816,3
3,88ab8d9,3
4,cee8c0f,2
